# Phase 1: Tokenization & Text Preprocessing

Welcome to the first notebook of your NLP learning journey! This notebook is organized into three distinct parts to help you build a deep, end-to-end understanding of text preprocessing:

1. **Part 1: From-Scratch Implementations**: Writing basic tokenization, sentence splitting, stemming, and BPE algorithms from scratch using standard Python string and regex operations.
2. **Part 2: Library-Based Preprocessing**: Replicating and comparing these steps using industry-standard libraries: **NLTK**, **spaCy**, and **Hugging Face Tokenizers**.
3. **Part 3: Hands-On Exercises**: Applying your skills to clean noisy social media text, compare stemming vs. lemmatization, and build a modular preprocessing pipeline.

---  
# Part 1: From-Scratch Implementations

Before touching any NLP library, let's build the core algorithms by hand. This will help you appreciate how standard packages handle complex edge cases under the hood.

## 1.1 Whitespace Tokenization

The simplest way to tokenize text is to split on whitespace characters. Let's implement this and examine its limitations.

In [ ]:
def tokenize_whitespace(text):
    """
    Splits text by whitespace characters (spaces, tabs, newlines).
    """
    # TODO: Implement whitespace splitting
    pass

In [ ]:
# Test cases to demonstrate limitations
sample_text = "NLP is state-of-the-art! However, it doesn't always work perfectly."
tokens = tokenize_whitespace(sample_text)
print("Whitespace Tokens:", tokens)

# Note how punctuation remains attached to words (e.g. "state-of-the-art!", "perfectly.").
# Note how contractions are kept as one token ("doesn't").

## 1.2 Regex-Based Word Tokenization

To handle punctuation and contractions more robustly, we can define token patterns using regular expressions (`re`).

In [ ]:
import re

def tokenize_regex(text):
    """
    Tokenizes text using regular expressions.
    Should separate punctuation from words, while keeping contractions (like "don't") intact.
    """
    # TODO: Define a pattern and extract tokens
    pattern = r"\w+(?:'\w+)?|[!\"#$%&'()*+,\-./:;<=>?@[\\\]^_`{|}~]"
    return re.findall(pattern, text)

In [ ]:
print("Regex Tokens:", tokenize_regex(sample_text))
# The punctuation marks should now be separated into their own individual tokens.

## 1.3 Simple Sentence Tokenization

Sentence tokenization splits a document into sentences. We cannot split simply on periods due to decimal numbers or abbreviations (like `Dr.`, `Mr.`, `e.g.`).

In [ ]:
def split_sentences(text):
    """
    Splits text into sentences, handling common sentence-ending punctuation (.?!)
    while attempting to avoid splitting on common abbreviations like Dr., Mr., etc.
    """
    # TODO: Implement basic sentence boundary detection
    pass

In [ ]:
sentence_sample = "Dr. Smith bought a laptop. It cost $1,000.50. What a deal!"
print("Sentences:", split_sentences(sentence_sample))

## 1.4 Basic Suffix Stemming

Stemming is the process of stripping suffixes (like `-ing`, `-ed`, `-ly`, `-s`) to reduce words to approximate roots.

In [ ]:
def simple_stemmer(word):
    """
    Reduces a word to its root by stripping common suffixes like -ing, -ed, -ly, -s.
    """
    # TODO: Implement basic suffix stripping rules
    pass

In [ ]:
test_words = ["connections", "connecting", "connected", "happily", "dogs"]
for w in test_words:
    print(f"{w} -> {simple_stemmer(w)}")

## 1.5 Byte Pair Encoding (BPE) from Scratch

BPE is a subword tokenization algorithm used by modern LLMs (like GPT). It starts with a character-level vocabulary and iteratively merges the most frequent pairs of adjacent characters/subwords.

In [ ]:
from collections import defaultdict

def get_vocab_counts(corpus):
    """
    Converts corpus of sentences into a dictionary of word frequencies, 
    representing words as tuples of characters with an end-of-word marker '</w>'.
    """
    vocab = defaultdict(int)
    for sentence in corpus:
        words = sentence.split()
        for word in words:
            char_tuple = tuple(list(word)) + ('</w>',)
            vocab[char_tuple] += 1
    return vocab

def get_stats(vocab):
    """
    Computes frequencies of all adjacent pairs of symbols in the vocabulary.
    """
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        for i in range(len(word)-1):
            pairs[(word[i], word[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab_in):
    """
    Merges all occurrences of the specified pair in the vocabulary.
    """
    vocab_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in vocab_in:
        word_str = ' '.join(word)
        word_merged = p.sub(''.join(pair), word_str)
        vocab_out[tuple(word_merged.split())] = vocab_in[word]
    return vocab_out

In [ ]:
# Run BPE merging for 10 iterations
corpus = [
    "low low low low low lower lower newer newer newer newer newer newer"
]
vocab = get_vocab_counts(corpus)
num_merges = 10

print("Initial Vocab:", vocab)
for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"Merge #{i+1}: {best} -> Vocab state: {vocab}")

---  
# Part 2: Library-Based Preprocessing

Now let's explore the industrial-strength libraries used in production and research: **NLTK**, **spaCy**, and the **Hugging Face tokenizers** library.

## 2.1 Environment Verification & Setup

Make sure you download the required resources for NLTK and spaCy models.

In [ ]:
import nltk
import spacy

# Download NLTK data (tokenizers and stopwords)
nltk.download('punkt')
nltk.download('stopwords')

# Download spaCy English model from terminal if you haven't already:
# !python -m spacy download en_core_web_sm

## 2.2 Word & Sentence Tokenization with NLTK

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize

nltk_words = word_tokenize(sample_text)
nltk_sentences = sent_tokenize(sample_text)

print("NLTK Words:", nltk_words)
print("NLTK Sentences:", nltk_sentences)

## 2.3 Tokenization, Lemmatization, and Stopwords with spaCy

In [ ]:
# Load the small English pipeline
nlp = spacy.load("en_core_web_sm")

doc = nlp(sample_text)

# Extract tokens, lemmas, and stopword status
spacy_data = []
for token in doc:
    spacy_data.append({
        "Text": token.text,
        "Lemma": token.lemma_,
        "Is Stopword": token.is_stop,
        "POS Tag": token.pos_
    })

import pandas as pd
df_spacy = pd.DataFrame(spacy_data)
df_spacy

## 2.4 Comparing Stemmers vs. Lemmatizers

Let's compare the outputs of NLTK Porter Stemmer vs. spaCy Lemmatizer.

In [ ]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

words_to_compare = ["connecting", "connection", "connections", "connected", "studies", "studying", "was", "am", "better"]

comparison = []
for word in words_to_compare:
    # Get stem
    stem = stemmer.stem(word)
    # Get lemma (via spaCy)
    doc_word = nlp(word)
    lemma = doc_word[0].lemma_
    
    comparison.append({
        "Word": word,
        "Porter Stem": stem,
        "spaCy Lemma": lemma
    })

df_compare = pd.DataFrame(comparison)
df_compare

## 2.5 Subword Tokenization with Hugging Face

Modern LLMs use BPE or WordPiece. Let's see how Hugging Face tokenizes text using a pre-trained subword tokenizer (like GPT-2).

In [ ]:
from transformers import GPT2Tokenizer

# Load pre-trained GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

tokens = tokenizer.tokenize(sample_text)
ids = tokenizer.convert_tokens_to_ids(tokens)

for t, i in zip(tokens, ids):
    print(f"Token: {t:<15} ID: {i}")

---  
# Part 3: Hands-On Exercises

Apply what you have learned about tokenization, text cleaning, stemming, and lemmatization to real-world tasks.

## Exercise 3.1: Stemming vs. Lemmatization Comparison

Evaluate the Porter Stemmer and spaCy Lemmatizer on a small corpus of sentences. Identify cases of over-stemming (stripping too much, changing meaning) and under-stemming (failing to strip suffixes to group words together).

In [ ]:
# Define sentences with complex word forms
sentences = [
    "The children are playing happily in the garden, having fun with their toys.",
    "The studies showed that studying regularly produces better study results.",
    "He was running, jumps over the fence, and then walked slowly home.",
    "The items were bought at a discount, but some were broken or damaged."
]

# TODO: Tokenize, stem, and lemmatize all words in these sentences. 
# Print them side-by-side or build a comparison table.

## Exercise 3.2: Noisy Social Media Cleaning Pipeline

Social media text is notorious for noisy elements like URLs, user mentions, hashtags, HTML entities, and excessive spacing. Build a regex cleaning pipeline to normalize these elements.

In [ ]:
tweets = [
    "Check out this amazing article! https://t.co/xyz123 @user123 #NLP #AI is the future!!!",
    "RT @another_user: Python is &lt;great&gt; for data science. Go to http://python.org",
    "I    love   natural    language    processing... it's so cool!!!"
]

def clean_tweet(text):
    """
    Cleans a tweet by:
    1. Removing URLs
    2. Removing user mentions (@username)
    3. Stripping/converting HTML entities (like &lt; to <)
    4. Normalizing whitespace (reducing multiple spaces to single space)
    """
    # TODO: Implement cleaning rules using re
    pass

for tweet in tweets:
    print("Original:", tweet)
    print("Cleaned: ", clean_tweet(tweet))
    print("-" * 50)

## Exercise 3.3: Complete Preprocessing Pipeline

Write a single function `preprocess_text` that acts as a pipeline. It should clean text, tokenize, remove stopwords, and lemmatize.

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    """
    Takes raw text, cleans it, tokenizes, removes stopwords/punctuation, 
    and returns a list of lemmatized tokens in lowercase.
    """
    # TODO: Implement complete preprocessing pipeline
    pass

In [ ]:
raw_doc = """The Quick Brown Fox jumps over the lazy dog! 
Visit http://example.com/fox for more details. @fox_fan club."""

print("Preprocessed:", preprocess_text(raw_doc))
# Expected output should contain only meaningful lemmas, like ['quick', 'brown', 'fox', 'jump', 'lazy', 'dog', 'visit', 'detail', 'club']